<a href="https://colab.research.google.com/github/davimoljo/Campe-Website/blob/master/C%C3%B3pia_de_Aula_8_Homografia_com_RANSAC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Criando imagens panorâmicas no OpenCV utilizando o RANSAC

Neste tutorial veremos como alinhar duas imagens para criar uma panorâmica no OpenCV.

Começaremos importando as bibliotecas e criando uma função para extrair as *features* de uma imagem.

In [ ]:
import numpy as np
import imutils
import cv2 as cv
from google.colab.patches import cv2_imshow

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1 - Extração de *features* (exercício)

Criamos uma função para extrair as features de uma imagem. Utilizaremos o SIFT inicialmente.

In [ ]:
def detectAndDescribe(image):
		gray = cv.cvtColor(image, cv.COLOR_BGR2GRAY)

		sift = cv.SIFT_create()
		kps, desc = sift.detectAndCompute(gray, None)

		# converte para NumPy
		kps = np.float32([kp.pt for kp in kps])

		return (kps, desc)

## 2 - Estabelecimento de correspondências

Em seguida criamos uma função para estabelecer correspondências entre as *features*. Essa função recebe os pontos e as *features* extraídas pela função anterior. A função pode utilizar o teste da razão para evitar falsos positivos. Basta definir o valor de `ratio` diferente de 0. Observe que norma utilizada  para comparar os descritores deve ser passada como parâmetro de acordo com o tipo de descritor.

In [ ]:
def matchKeypoints(kpsA, kpsB, featuresA, featuresB, normType, ratio):

		# Cria o objeto responsável pelas correspondências
		matcher = cv.BFMatcher(normType = normType)
		matches = []

		if ratio > 0:
				# Encontra as duas melhores correspondências para cada ponto
				rawMatches = matcher.knnMatch(featuresA, featuresB, 2)

				#percorre todos os matches
				for m in rawMatches:
						# Aplica o teste da razão das duas menores distâncias para eliminar falsos positivos
						if len(m) == 2 and m[0].distance < m[1].distance * ratio:
								matches.append((m[0].trainIdx, m[0].queryIdx))
		else:
				matches = matcher.match(featuresA, featuresB)

		return matches

## 3 - Computa a homografia

Criamos uma função que computa a homografia, utilizando o RANSAC, a partir das correspondências entre as imagens. O método `findHomography` é responsável por computar a homografia entre as imagens a partir das correspondências. Esse método tem como principais parâmetros:
* Os *keypoints* de cada imagem
* `method`: o método a ser empregado (opcional, por *default* é 0, indicando mínimos quadrados com todas as correspondências. Para utilizar o RANSAC, utiliza-se a constante `cv.RANSAC`)
* `ransacReprojThreshold`: o limiar para considerar uma correspondência como *inlier* no RANSAC (opcional, por *default* é 3)
* `maxIters`: número máximo de iterações do RANSAC (opcional, por *default* é 2000)
* `confidence`: nível de confiança do RANSAC (opcional, por *default* é 0.995)

O método retorna a homografia e o status de cada correspondência, sendo 0 para *outlier* e 1 para *inlier*.

In [ ]:
def computeHomography(matches, kpsA, kpsB, reprojThresh):
    # Caso tenha pelo menos 4 correspondências
		if len(matches) > 4:
			# Constroi os dois conjuntos de pontos
			ptsA = np.float32([kpsA[i] for (_, i) in matches])
			ptsB = np.float32([kpsB[i] for (i, _) in matches])
			# Computa a homografia entre os pontos utilizando o RANSAC
			(H, status) = cv.findHomography(ptsA, ptsB, cv.RANSAC, reprojThresh)

			# Retorna a homografia, os matches e o respectivo status (1 para inlier, 0 para outlier)
			return (matches, H, status)

		# Caso contrário, a homografia não é computada
		return None

## 4 - Visualizando as correspondências

Vamos definir uma função para gerar a visualização das correspondências.

In [ ]:
def drawMatches(imageA, imageB, kpsA, kpsB, matches, status):
		# Inicializa a imagem de visualização
		(hA, wA) = imageA.shape[:2]
		(hB, wB) = imageB.shape[:2]
		vis = np.zeros((max(hA, hB), wA + wB, 3), dtype="uint8")
		vis[0:hA, 0:wA] = imageA
		vis[0:hB, wA:] = imageB
		# Percorre a lista de correspondências
		for ((trainIdx, queryIdx), s) in zip(matches, status):
			# Desenha as correspondências que foram identificadas como inliers na homografia
			if s == 1:
				# Desenha as correspondências
				ptA = (int(kpsA[queryIdx][0]), int(kpsA[queryIdx][1]))
				ptB = (int(kpsB[trainIdx][0]) + wA, int(kpsB[trainIdx][1]))
				cv.line(vis, ptA, ptB, (0, 255, 0), 1)
		# Retorna a visualização
		return vis

## 5 - Alinhando as imagens

Definimos a função que alinha as imagens e gera a imagem composta. Imagens são passadas como uma lista. Função também recebe os parâmetros referentes aos limiares do RANSAC e um parâmetro que determina se as correspondências serão mostradas.

In [ ]:
def stitch(images, ratio=0.75, normType = cv.NORM_L2, reprojThresh=4.0, showMatches=False):
		# Separa as imagens e calcula as features
		(imageB, imageA) = images
		(kpsA, featuresA) = detectAndDescribe(imageA)
		(kpsB, featuresB) = detectAndDescribe(imageB)

		# estabelece as correspondências
		M = matchKeypoints(kpsA, kpsB,
			featuresA, featuresB, normType, ratio)

		# Se não há correspondências suficientes, não é possível criar a imagem
		if M is None:
			return None

    # Caso contrário, estima a homografia e aplica a transformação para alinhar as imagens
		(matches, H, status) = computeHomography(M, kpsA, kpsB, reprojThresh)
		result = cv.warpPerspective(imageA, H, (imageA.shape[1] + imageB.shape[1], imageA.shape[0]))
		result[0:imageB.shape[0], 0:imageB.shape[1]] = imageB

		# Caso a opção de visualizar os matches tenha sido definida, mostra as correspondências
		if showMatches:
			vis = drawMatches(imageA, imageB, kpsA, kpsB, matches, status)

      # Retorna a imagem resultante e a visualização das correspondências
			return (result, vis)

		# Retorna somente a imagem resultante do alinhamento
		return result

NameError: name 'cv' is not defined

## 6 - Exercícios

1. Utilizando as funções definidas anteriormente e as imagens sugeridas ([esquerda](https://drive.google.com/file/d/12LyP6tkXcTA4hWqEbgtyxKBwqZtClK8-/view?usp=sharing) e [direita](https://drive.google.com/file/d/1QrugJwTHVfwq-OclOL1oGDz8IPEzMb_q/view?usp=sharing)), faça o alinhamento e mostre o resultado.

In [ ]:
# Insira seu código aqui

Faça algumas variações no método e avalie os resultados (altere parâmetros, altere o tipo de descritor, tente remover o RANSAC, etc)